# Structured Output in LangChain

Structured output allows you to turn the "chaos" of natural language into reliable, typed Python objects. By using the `.with_structured_output()` method, you eliminate the need for regular expressions or complex "Please return JSON" prompts.

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.


### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

---

## 1. The Core Method: `.with_structured_output()`

This is a high-level method available on most Chat Models (OpenAI, Anthropic, Gemini, etc.). It wraps the model and ensures that the output conforms exactly to your provided schema.



### Basic Implementation
```python
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# 1. Define your data structure
class Book(BaseModel):
    title: str = Field(description="The title of the book")
    author: str = Field(description="The author's name")
    year: int = Field(description="The publication year")

# 2. Initialize and wrap the model
llm = ChatOpenAI(model="gpt-4o")
structured_llm = llm.with_structured_output(Book)

# 3. Use the model (returns a 'Book' object, not a string)
response = structured_llm.invoke("Tell me about The Great Gatsby")
print(f"{response.title} was written by {response.author} in {response.year}.")
```

## 2. Advanced: Nested & Complex Schemas
Structured output is most powerful when dealing with lists or nested objects. You can define a "parent" Pydantic model that contains a "child" list of models.

```python
from typing import List

class Ingredient(BaseModel):
    name: str
    amount: str

class Recipe(BaseModel):
    dish_name: str
    servings: int
    ingredients: List[Ingredient] # Nested list of objects

structured_recipe_llm = llm.with_structured_output(Recipe)
```

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
from langchain.chat_models import init_chat_model
model=init_chat_model("gpt-4.1")
model

ChatOpenAI(profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000024212067140>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000242123FC2F0>, root_client=<openai.OpenAI object at 0x000002421177E960>, root_async_client=<openai.AsyncOpenAI object at 0x0000024211D66240>, model_name='gpt-4.1', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [3]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [4]:
model_with_str_output = model.with_structured_output(Movie)
model_with_str_output

RunnableBinding(bound=ChatOpenAI(profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000024212067140>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000242123FC2F0>, root_client=<openai.OpenAI object at 0x000002421177E960>, root_async_client=<openai.AsyncOpenAI object at 0x0000024211D66240>, model_name='gpt-4.1', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True), kwargs={'response_format': <class '__main__.Movie'>, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema', 'strict': N

In [6]:
model.invoke("Provide details about the movie Avatar")

AIMessage(content='Certainly! Here are details about the movie **"Avatar"**:\n\n---\n\n### **Avatar (2009)**\n\n**Director:**  \nJames Cameron\n\n**Producer(s):**  \nJames Cameron, Jon Landau\n\n**Writer:**  \nJames Cameron\n\n**Starring:**  \n- Sam Worthington as Jake Sully  \n- Zoe Saldana as Neytiri  \n- Sigourney Weaver as Dr. Grace Augustine  \n- Stephen Lang as Colonel Miles Quaritch  \n- Giovanni Ribisi as Parker Selfridge  \n- Michelle Rodriguez as Trudy Chacón  \n- CCH Pounder as Mo\'at  \n\n**Genre:**  \nScience fiction, Adventure, Action\n\n**Running Time:**  \n162 minutes (theatrical version)\n\n**Release Date:**  \nDecember 18, 2009\n\n---\n\n### **Synopsis**\n\n*Avatar* is set in the mid-22nd century, when humans are colonizing Pandora, a lush and habitable moon orbiting the star system Alpha Centauri. The expansion of Earth\'s population has led to a depletion of natural resources, prompting the Resources Development Administration (RDA) to mine Pandora for a valuable mi

In [ ]:
model_with_str_output.invoke("Provide details about the movie Avatar")

Movie(title='Avatar', year=2009, director='James Cameron', rating=7.8)

## Message output alongside parsed Structure 

In [9]:
class Movie(BaseModel):
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="This year the movie was released")
    director:str=Field(...,description="The director of the movie")
    rating:float=Field(...,description="The movies rating out of 10")

model_with_str_output = model.with_structured_output(Movie,include_raw=True)
model_with_str_output.invoke("Provide details about the movie Avatar")

{'raw': AIMessage(content='{"title":"Avatar","year":2009,"director":"James Cameron","rating":7.8}', additional_kwargs={'parsed': Movie(title='Avatar', year=2009, director='James Cameron', rating=7.8), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 117, 'total_tokens': 138, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_1a2c4a5ede', 'id': 'chatcmpl-D25mMSiytxFikSfTeqSn6vxtb7BMD', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019bf7ef-b7b0-7a80-885b-1dd30a287cc6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 21, 'total_tokens': 138, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'

## Nested Structure

In [12]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title : str
    year : int 
    cast : list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_str_output = model.with_structured_output(MovieDetails)
movie_details = model_with_str_output.invoke("Provide details about move Avatar")
print(movie_details)

print(movie_details.title)

title='Avatar' year=2009 cast=[Actor(name='Sam Worthington', role='Jake Sully'), Actor(name='Zoe Saldana', role='Neytiri'), Actor(name='Sigourney Weaver', role='Dr. Grace Augustine'), Actor(name='Stephen Lang', role='Colonel Miles Quaritch'), Actor(name='Giovanni Ribisi', role='Parker Selfridge')] genres=['Action', 'Adventure', 'Sci-Fi'] budget=237.0
Avatar


### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [ ]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """ A Movie with Details"""
    title: Annotated[str, ..., "The title of the Movie" ]
    year: Annotated[int,..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_str_output = model.with_structured_output(MovieDict)
movie_details = model_with_str_output.invoke("Tell about the details of Avatar movie")
movie_details

{'title': 'Avatar', 'year': 2009, 'director': 'James Cameron', 'rating': 7.8}

In [19]:
movie_details['title']

'Avatar'

In [20]:
model.profile

{'max_input_tokens': 1047576,
 'max_output_tokens': 32768,
 'image_inputs': True,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'structured_output': True,
 'image_url_inputs': True,
 'pdf_inputs': True,
 'pdf_tool_message': True,
 'image_tool_message': True,
 'tool_choice': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [21]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='dfc4c9c3-af4d-4b91-95e4-a75851e2e71e'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 804, 'prompt_tokens': 204, 'total_tokens': 1008, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-D26EnJC75ydSaLAXGldzRwfrV4qJH', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019bf80a-a280-79e3-8b15-243ff5bf18be-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 204, 'outpu

In [22]:
result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [23]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [24]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')